In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error

RESULTS_DIR = r"C:\Users\Admin\PycharmProjects\Monetary_transmission\project\results"
TARGET_COLS = ["CPI", "EXR"]
WINDOWS = ["W1", "W2", "W3", "W4", "W5"]

# Display name -> the suffix used in this project's predictions_<suffix>.csv files.
MODEL_SOURCES = {
    "VECM": "var_vecm",
    "ARIMA": "arima",
    "FFNN": "ffnn",
    "TDNN": "tdnn",
    "LSTM": "lstm",
}

In [2]:
def load_predictions(results_dir: str) -> dict:
    """Load whichever base-model prediction files are available, skipping the rest."""
    loaded = {}
    for name, suffix in MODEL_SOURCES.items():
        path = os.path.join(results_dir, f"predictions_{suffix}.csv")
        if os.path.exists(path):
            loaded[name] = pd.read_csv(path, parse_dates=["date"])
        else:
            print(f"[skip] {path} not found — excluding {name} from the ensemble")
    return loaded


def load_metrics(results_dir: str, model_names: list) -> dict:
    """Load each model's own metrics_*.csv, used only for the weighted-average weights."""
    metrics = {}
    for name in model_names:
        path = os.path.join(results_dir, f"metrics_{MODEL_SOURCES[name]}.csv")
        metrics[name] = pd.read_csv(path)
    return metrics


predictions = load_predictions(RESULTS_DIR)
metrics = load_metrics(RESULTS_DIR, list(predictions.keys()))
print(f"Models available for ensembling: {list(predictions.keys())}")

predictions = load_predictions(RESULTS_DIR)
metrics = load_metrics(RESULTS_DIR, list(predictions.keys()))
print(f"Models available for ensembling: {list(predictions.keys())}")

for name, df in predictions.items():
    print(f"  {name}: windows={sorted(df['window'].unique())}, "
          f"dates={df['date'].min().date()}..{df['date'].max().date()}, rows={len(df)}")

Models available for ensembling: ['VECM', 'ARIMA', 'FFNN', 'TDNN', 'LSTM']
Models available for ensembling: ['VECM', 'ARIMA', 'FFNN', 'TDNN', 'LSTM']
  VECM: windows=['W1', 'W2', 'W3', 'W4', 'W5'], dates=2007-09-01..2025-10-01, rows=160
  ARIMA: windows=['W1', 'W2', 'W3', 'W4', 'W5'], dates=2007-09-01..2025-10-01, rows=160
  FFNN: windows=['W1', 'W2', 'W3', 'W4', 'W5'], dates=2007-12-01..2025-10-01, rows=104
  TDNN: windows=['W1', 'W2', 'W3', 'W4', 'W5'], dates=2007-05-01..2025-10-01, rows=220
  LSTM: windows=['W1', 'W2', 'W3', 'W4', 'W5'], dates=2007-10-01..2025-10-01, rows=150


In [3]:
def align_predictions(loaded: dict, window: str, target: str) -> pd.DataFrame:
    """Inner-join every available model's predictions for one window/target on date."""
    wide = None
    for name, df in loaded.items():
        sub = df[(df["window"] == window) & (df["target"] == target)][["date", "actual", "predicted"]]
        if sub.empty:
            continue
        sub = sub.rename(columns={"predicted": name}).set_index("date")
        if wide is None:
            wide = sub
        else:
            wide = wide.join(sub[[name]], how="inner")
    return wide.dropna() if wide is not None else pd.DataFrame()


def inverse_rmse_weights(metrics: dict, model_names: list, window: str, target: str) -> dict:
    """Normalize each model's own reported RMSE (its own test split) into a weight."""
    rmses = {}
    for name in model_names:
        row = metrics[name][(metrics[name]["window"] == window) & (metrics[name]["target"] == target)]
        if not row.empty and row["RMSE"].iloc[0] > 0:
            rmses[name] = row["RMSE"].iloc[0]
    inverse = {name: 1.0 / r for name, r in rmses.items()}
    total = sum(inverse.values())
    return {name: w / total for name, w in inverse.items()}


def stacked_forecast(wide: pd.DataFrame, model_names: list, label: str = "") -> np.ndarray:
    """
    Non-negative Ridge meta-learner over the base models' predictions, scored via
    leave-one-out CV so every stacked value is genuinely out-of-sample even though
    the overlap sample per window can be small.

    With only a handful of test points and one feature per base model, LOO-CV can
    still look deceptively strong (each fold fits n-1 points with len(model_names)
    free coefficients, which is nearly saturated). Require at least 3x as many points
    as features before trusting a meta-fit at all; otherwise fall back to the simple
    average, which has no such failure mode.
    """
    X, y = wide[model_names].values, wide["actual"].values
    if len(y) < 3 * len(model_names):
        print(f"  [stacked fallback] {label}: only {len(y)} points for {len(model_names)} models — using simple average instead")
        return wide[model_names].mean(axis=1).values
    return cross_val_predict(Ridge(alpha=5.0, positive=True), X, y, cv=LeaveOneOut())

In [4]:
model_names = list(predictions.keys())
results = {}

for window in WINDOWS:
    for target in TARGET_COLS:
        wide = align_predictions(predictions, window, target)
        present = [m for m in model_names if m in wide.columns]
        if wide.empty or len(present) < 2:
            print(f"[skip] {window}/{target}: only {len(present)} overlapping model(s) ({present})")
            continue

        simple = wide[present].mean(axis=1).values
        weights = inverse_rmse_weights(metrics, present, window, target)
        weighted = sum(wide[m].values * weights[m] for m in present)
        stacked = stacked_forecast(wide, present, label=f"{window}/{target}")

        actual = wide["actual"].values
        results[(window, target)] = {
            "dates": wide.index, "actual": actual, "n_models": len(present), "models": present,
            "simple": simple, "weighted": weighted, "stacked": stacked,
        }
        print(f"{window}/{target}: {len(present)} models, {len(actual)} overlapping test points")

if not results:
    raise RuntimeError(
        "No window/target produced an overlapping ensemble — check the per-model "
        "window/date diagnostics printed in cell 2 for mismatched labels or date ranges."
    )

  [stacked fallback] W1/CPI: only 6 points for 5 models — using simple average instead
W1/CPI: 5 models, 6 overlapping test points
  [stacked fallback] W1/EXR: only 6 points for 5 models — using simple average instead
W1/EXR: 5 models, 6 overlapping test points
  [stacked fallback] W2/CPI: only 9 points for 5 models — using simple average instead
W2/CPI: 5 models, 9 overlapping test points
  [stacked fallback] W2/EXR: only 9 points for 5 models — using simple average instead
W2/EXR: 5 models, 9 overlapping test points
  [stacked fallback] W3/CPI: only 6 points for 5 models — using simple average instead
W3/CPI: 5 models, 6 overlapping test points
  [stacked fallback] W3/EXR: only 6 points for 5 models — using simple average instead
W3/EXR: 5 models, 6 overlapping test points
  [stacked fallback] W4/CPI: only 4 points for 5 models — using simple average instead
W4/CPI: 5 models, 4 overlapping test points
  [stacked fallback] W4/EXR: only 4 points for 5 models — using simple average inst

In [5]:
def summarize(variant: str) -> pd.DataFrame:
    """RMSE/MAE per window/target for one ensembling variant."""
    rows = []
    for (window, target), r in results.items():
        pred = r[variant]
        rows.append({
            "window": window, "target": target,
            "RMSE": float(np.sqrt(mean_squared_error(r["actual"], pred))),
            "MAE": float(mean_absolute_error(r["actual"], pred)),
        })
    return pd.DataFrame(rows)


summary_simple = summarize("simple")
summary_weighted = summarize("weighted")
summary_stacked = summarize("stacked")

comparison = pd.concat({
    "Simple": summary_simple.set_index(["window", "target"])["RMSE"],
    "Weighted": summary_weighted.set_index(["window", "target"])["RMSE"],
    "Stacked": summary_stacked.set_index(["window", "target"])["RMSE"],
}, axis=1)
comparison.round(4)

Simple  Weighted  Stacked
window target                            
W1     CPI      9.1453    8.1979   9.1453
       EXR      4.0421    3.7780   4.0421
W2     CPI      2.5665    1.8690   2.5665
       EXR      8.6751    8.1352   8.6751
W3     CPI     13.8311   12.2054  13.8311
       EXR      8.6110    1.7922   8.6110
W4     CPI      3.0428    1.2762   3.0428
       EXR      7.3654    3.7093   7.3654
W5     CPI     17.4027   11.0893   8.2528
       EXR     27.1185   13.0610   7.6194

In [6]:
os.makedirs("../results", exist_ok=True)

# Standardized long-format export consumed by the model-comparison pipeline —
# one file per ensembling variant, registered as separate models downstream.
summary_simple.to_csv("../results/metrics_hybrid_ensemble_simple.csv", index=False)
summary_weighted.to_csv("../results/metrics_hybrid_ensemble_weighted.csv", index=False)
summary_stacked.to_csv("../results/metrics_hybrid_ensemble_stacked.csv", index=False)

# Combined raw predictions across all three variants, for closer inspection.
pred_rows = [
    {
        "window": window, "target": target, "date": date, "actual": actual,
        "simple": simple, "weighted": weighted, "stacked": stacked,
        "n_models": r["n_models"], "models": "|".join(r["models"]),
    }
    for (window, target), r in results.items()
    for date, actual, simple, weighted, stacked in zip(
        r["dates"], r["actual"], r["simple"], r["weighted"], r["stacked"]
    )
]
pd.DataFrame(pred_rows).to_csv("../results/predictions_hybrid_ensemble.csv", index=False)